[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/notebooks/07_eyes_convolutions.ipynb)

# ⚒️ Act II · Quest 07 — Eyes — Convolutions

*Give your network vision: filters, feature maps, and a trained glyph classifier you can dissect.*

⬅️ [06_feeding_the_beast](06_feeding_the_beast.ipynb)  •  [08_memory_sequences](08_memory_sequences.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Why a `Linear` layer is bad at seeing

Flatten a 20×20 glyph into 400 numbers and a `Linear` layer treats pixel (3,3) and pixel (3,4)
as *totally unrelated* inputs. Move the glyph one pixel and every weight is wrong.

A **convolution** slides one small filter across the whole image:
- **locality** — nearby pixels are processed together,
- **weight sharing** — the same detector works at every position (shift a glyph → same response, shifted),
- **efficiency** — a 3×3 filter is 9 weights, not 400.

### A filter is a pattern-detector. Watch one hunt for diagonals:

In [ ]:
# --- The Glyph dataset: ✕ ◯ ┼ ╱  (self-contained, no torchvision needed) ----
GLYPHS = ["cross", "ring", "plus", "slash"]

def _draw_glyph(cls, size=20, rng=None):
    rng = rng or torch.Generator().manual_seed(0)
    img = torch.zeros(size, size)
    ys, xs = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    jx = int(torch.randint(-2, 3, (1,), generator=rng))   # random jitter
    jy = int(torch.randint(-2, 3, (1,), generator=rng))
    c = size // 2
    if cls == 0:    # cross ✕ : two diagonals
        img[((xs - ys).abs() + (jx - jy)).abs() <= 1] = 1.0
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    elif cls == 1:  # ring ◯
        r2 = (xs - c - jx) ** 2 + (ys - c - jy) ** 2
        img[(r2 >= 25) & (r2 <= 49)] = 1.0
    elif cls == 2:  # plus ┼
        img[(ys - c - jy).abs() <= 1] = 1.0
        img[(xs - c - jx).abs() <= 1] = 1.0
    else:           # slash ╱ : one diagonal
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    img = img + 0.08 * torch.randn(size, size, generator=rng)
    return img.clamp(0, 1)

def make_glyphs(n_per_class=300, size=20, seed=0):
    rng = torch.Generator().manual_seed(seed)
    X = torch.stack([_draw_glyph(c, size, rng) for c in range(4) for _ in range(n_per_class)])
    y = torch.arange(4).repeat_interleave(n_per_class)
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].unsqueeze(1), y[perm]   # (N, 1, 20, 20), (N,)

In [ ]:
X, y = make_glyphs(n_per_class=200)

# Hand-forged filters
diag = torch.tensor([[2., -1, -1], [-1, 2., -1], [-1, -1, 2.]])     # ↘ detector
anti = diag.flip(1)                                                  # ↙ detector
hbar = torch.tensor([[-1., -1, -1], [2., 2., 2.], [-1., -1, -1]])   # ─ detector
bank = torch.stack([diag, anti, hbar]).unsqueeze(1)                  # (3,1,3,3)

sample = X[y == 0][:1]     # a cross ✕ (made of two diagonals)
maps = F.conv2d(sample, bank, padding=1)

fig, ax = plt.subplots(1, 4, figsize=(11, 3))
ax[0].imshow(sample[0, 0], cmap="gray"); ax[0].set_title("input: cross ✕")
for i, name in enumerate(["↘ detector", "↙ detector", "─ detector"]):
    ax[i + 1].imshow(maps[0, i], cmap="inferno"); ax[i + 1].set_title(name)
for a in ax: a.axis("off")
plt.show()
print("the ↘ and ↙ maps light up along the arms; the ─ map mostly doesn't. Detection!")

### Conv arithmetic — the formula you'll compute in your head forever

For input size \(N\), kernel \(K\), padding \(P\), stride \(S\):

\[ \text{out} = \left\lfloor \frac{N + 2P - K}{S} \right\rfloor + 1 \]

And parameters of `Conv2d(c_in, c_out, K)` = \(c_{out} \cdot (c_{in} \cdot K \cdot K + 1)\).

In [ ]:
for (cin, cout, k, p, s) in [(1, 16, 3, 1, 1), (16, 32, 3, 1, 2), (1, 8, 5, 0, 1)]:
    conv = nn.Conv2d(cin, cout, k, padding=p, stride=s)
    out = conv(torch.randn(1, cin, 20, 20))
    n_params = sum(pp.numel() for pp in conv.parameters())
    print(f"Conv2d({cin:2d},{cout:2d},k={k},p={p},s={s}): 20 -> {out.shape[-1]:2d}   params={n_params}")

### Build & train GlyphNet

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

ds = TensorDataset(X, y)
train_ds, test_ds = random_split(ds, [640, 160], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

class GlyphNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 20 -> 10
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 10 -> 5
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 5 * 5, 64), nn.ReLU(), nn.Linear(64, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

model = GlyphNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(loader):
    model.eval(); hits = n = 0
    with torch.no_grad():
        for xb, yb in loader:
            hits += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item(); n += len(yb)
    return hits / n

for epoch in range(4):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    print(f"epoch {epoch}: test accuracy {accuracy(test_loader)*100:.1f}%")

### Dissect it: what did the filters *become*?

In [ ]:
# 1) learned first-layer filters
W = model.features[0].weight.detach().cpu()
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax[i // 8, i % 8].imshow(W[i, 0], cmap="RdBu"); ax[i // 8, i % 8].axis("off")
plt.suptitle("Learned filters — gradient descent invented its own edge detectors"); plt.show()

# 2) feature maps for one glyph
sample = X[y == 1][:1].to(device)       # a ring ◯
with torch.no_grad():
    fmaps = model.features[0](sample).cpu()[0]
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax[i // 8, i % 8].imshow(fmaps[i], cmap="inferno"); ax[i // 8, i % 8].axis("off")
plt.suptitle("Feature maps: 16 different 'views' of one ring"); plt.show()

In [ ]:
# 3) confusion matrix — WHERE does it fail?
model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.append(model(xb.to(device)).argmax(1).cpu()); trues.append(yb)
preds, trues = torch.cat(preds), torch.cat(trues)
cm = torch.zeros(4, 4, dtype=torch.int)
for t, p in zip(trues, preds):
    cm[t, p] += 1
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(GLYPHS); ax.set_yticks(range(4)); ax.set_yticklabels(GLYPHS)
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() // 2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("confusion matrix")
plt.show()
print("note: cross ✕ vs slash ╱ share a diagonal — the most confusable pair. Errors make sense!")

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda n: n == 10,
    "N=20, K=3, P=1, S=2 -> floor((20 + 2 - 3)/2) + 1 = 10")
_register("param_count", 10,
    lambda n: n == 4640,
    "Conv2d(16, 32, 3): 32 * (16*3*3 + 1) = 4640")
_register("glyph_92", 20,
    lambda m: isinstance(m, nn.Module) and (lambda: (
        [m.eval()] and sum((m(xb.to(device)).argmax(1).cpu() == yb).sum().item() for xb, yb in test_loader)
        / sum(len(yb) for _, yb in test_loader) >= 0.92))(),
    "reach >= 92% test accuracy — train longer, add a conv block, or augment. Submit the model.")
_register("why_share", 10,
    lambda s: "shift" in s.lower() or "translation" in s.lower() or "position" in s.lower() or "anywhere" in s.lower(),
    "one sentence: weight sharing means the same detector works regardless of ____ in the image")

In [ ]:
check("warmup", 10)

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `param_count` (10 XP) — how many parameters in `Conv2d(16, 32, 3)`? Compute by hand first.
- `glyph_92` (20 XP) — push test accuracy to ≥92% and submit the model.
- `why_share` (10 XP) — in one sentence: why is weight sharing the right idea for images?

In [ ]:
# ⚔️ your attempts here...

# xp_report()